In [1]:
import numpy as np
import scipy.stats as stats
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
from utils import performance_dict, sigmoid
from tqdm import tqdm

file = '/home/server/Projects/data/AKI/tabular_preop.csv'
df = pd.read_csv(file)

In [2]:
class bootstrap_split(object):
    def __init__(self, df):
        self.df = df
        self.i_df = 0   # cycling index out of five
        self.i = 0      # total index out of 25
        self.df_fifths = [] 

    def __iter__(self):
        return self
    def __next__(self):
        return self.next()
    
    def df_to_arrays(self, df):
        remove_cols = ['op_id', 'aki', 'aki_boolean', 'aki_positive']
        # remove_cols += [col for col in df.columns if '_isna' in col]
        
        X = df.drop(columns=remove_cols, errors='ignore').values
        y = df['aki'].values
        y_binary = df['aki_boolean'].values
        return X, y, y_binary
    
    def upsample(self, df):
        df_majority = df[df["aki_boolean"] == 0]
        df_minority = df[df["aki_boolean"] == 1]

        df_minority_upsampled = resample(df_minority,
                    replace=True,  # Sample with replacement
                    n_samples=len(df_majority),  # Match majority class size
                    random_state=42)

        # Combine upsampled minority with majority class
        df_balanced = pd.concat([df_majority, df_minority_upsampled])
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
        return df_balanced

    def next(self):
        if self.i == 25:
            raise StopIteration()
        elif self.i % 5 == 0:
            self.i_df = 0
            self.df_fifths = [] 
            df_remainder = self.df
            for remaining_fifths in range(5, 1, -1):
                df_remainder, df_fifth = train_test_split(df_remainder, 
                                            test_size=(1.0/remaining_fifths), 
                                            random_state=42 + (self.i // 5), 
                                            stratify=df_remainder["aki_boolean"])
                self.df_fifths.append(df_fifth)
            self.df_fifths.append(df_remainder)
        df_temp = self.df_fifths.pop(self.i_df)
        X_test, y_test, y_binary_test = self.df_to_arrays(df_temp)
        X_train, y_train, y_binary_train = self.df_to_arrays(self.upsample(pd.concat(self.df_fifths)))
        self.df_fifths.insert(self.i_df, df_temp)
        self.i_df += 1
        self.i += 1
        return (X_test, y_test, y_binary_test), (X_train, y_train, y_binary_train)

def print_confidence_intervals(df_results):
    lows = []
    means = []
    highs = []
    for col in df_results.columns:
        mean = np.mean(df_results[col].values)
        sem = stats.sem(df_results[col].values)  # Standard error of the mean

        # Get 95% confidence interval
        confidence = 0.95
        n = len(df_results[col].values)
        dof = n - 1  # Degrees of freedom
        ci = stats.t.interval(confidence, dof, loc=mean, scale=sem)
        lows.append(ci[0])
        means.append(mean)
        highs.append(ci[1])
    for col in df_results.columns:
        print(col)
    print()
    for arr in [lows, means, highs]:
        for num in arr:
            print(f"{num:.4f}")
        print()


In [3]:
# -------------------- XGBoost Classification --------------------

import xgboost as xgb

df_results = pd.DataFrame()
for test, train in tqdm(bootstrap_split(df)):
    X_test, y_test, y_binary_test = test
    X_train, y_train, y_binary_train = train



    model = xgb.XGBClassifier(
            objective="binary:logistic",  # Binary classification (log loss)
            eval_metric="logloss",        # Logarithmic loss for better convergence
            use_label_encoder=False,      # Avoids unnecessary warnings
            n_estimators=1000,             # Number of boosting rounds
            learning_rate=0.1,            # Step size shrinkage
            max_depth=6,                  # Limits tree depth for regularization
            subsample=0.8,                # Prevents overfitting
            colsample_bytree=0.8,         # Reduces features per tree to avoid overfitting
            random_state=42
    )
    
    model.fit(X_train, y_binary_train)
    y_pred_binary = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    output_dict = performance_dict(y_binary_test, y_pred_binary, y_prob)



    if df_results.empty:
        df_results = pd.DataFrame(columns=output_dict.keys())
    df_results.loc[len(df_results)] = output_dict

print_confidence_intervals(df_results)
    

0it [00:00, ?it/s]

/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [07:41:09] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
1it [01:30, 90.70s/it]/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [07:42:40] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
2it [03:02, 91.36s/it]/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [07:44:11] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
3it [04:36, 92.73s/it]/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [07:45:46] WARNING: /wor

Precision
Sensitivity
Accuracy
rc_auc
pr_auc
Specificity
Negative Predictive Value
F1 Score

0.4538
0.1707
0.9383
0.7687
0.2549
0.9867
0.9495
0.2484

0.4648
0.1766
0.9388
0.7712
0.2604
0.9871
0.9498
0.2557

0.4758
0.1824
0.9393
0.7737
0.2659
0.9876
0.9502
0.2630



In [4]:
# -------------------- SVM VOTING ENSEMBLE CLASSIFICATION --------------------
from joblib import Parallel, delayed
from sklearn.svm import SVC

df_results = pd.DataFrame()
for test, train in tqdm(bootstrap_split(df)):
    X_test, y_test, y_binary_test = test
    X_train, y_train, y_binary_train = train


    def train_and_predict(X_batch, y_batch, X_test):
        svc = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
        svc.fit(X_batch, y_batch)
        y_pred = svc.predict(X_test)
        y_prob = svc.predict_proba(X_test)[:, 1]
        return y_pred, y_prob

    batch_size = 2000
    num_batches = int(np.ceil(len(X_train) / batch_size))

    # Parallel processing
    results = Parallel(n_jobs=-1)(delayed(train_and_predict)(
        X_train[i * batch_size : (i + 1) * batch_size], 
        y_binary_train[i * batch_size : (i + 1) * batch_size], 
        X_test
    ) for i in tqdm(range(num_batches)))

    # Aggregate results
    y_pred_sum = np.sum([res[0] for res in results], axis=0)
    y_prob_sum = np.sum([res[1] for res in results], axis=0)

    y_pred = (y_pred_sum / num_batches) > 0.5
    y_prob = y_pred_sum / num_batches
    output_dict = performance_dict(y_binary_test, y_pred, y_prob)



    if df_results.empty:
        df_results = pd.DataFrame(columns=output_dict.keys())
    df_results.loc[len(df_results)] = output_dict
    
    # in case of outage, since this run takes so long
    temp_file = '/home/server/Projects/data/AKI/bootstrap/temp.csv'
    df_results.to_csv(temp_file)

print_confidence_intervals(df_results)
    

100%|██████████| 45/45 [03:53<00:00,  5.19s/it]
25it [2:17:08, 329.14s/it]

Precision
Sensitivity
Accuracy
rc_auc
pr_auc
Specificity
Negative Predictive Value
F1 Score

0.1457
0.6506
0.7507
0.7698
0.3113
0.7566
0.9717
0.2381

0.1474
0.6580
0.7529
0.7737
0.3161
0.7589
0.9723
0.2408

0.1491
0.6653
0.7550
0.7776
0.3210
0.7612
0.9728
0.2434



In [3]:
file = '/home/server/Projects/data/AKI/tabular_preop.csv'
df = pd.read_csv(file)

# -------------------- MLP CLASSIFICATION --------------------

from sklearn.neural_network import MLPClassifier

df_results = pd.DataFrame()
for test, train in tqdm(bootstrap_split(df)):
    X_test, y_test, y_binary_test = test
    X_train, y_train, y_binary_train = train


    arch = (8, 8, 4, 4)

    model = MLPClassifier(random_state=42, max_iter=1000, hidden_layer_sizes=arch)
    model.fit(X_train, y_binary_train)
    y_pred_binary = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    output_dict = performance_dict(y_binary_test, y_pred_binary, y_prob)



    if df_results.empty:
        df_results = pd.DataFrame(columns=output_dict.keys())
    df_results.loc[len(df_results)] = output_dict

print_confidence_intervals(df_results)
    

25it [24:08, 57.93s/it]

Precision
Sensitivity
Accuracy
rc_auc
pr_auc
Specificity
Negative Predictive Value
F1 Score

0.1650
0.7649
0.7519
0.8298
0.2687
0.7503
0.9808
0.2720

0.1685
0.7729
0.7586
0.8332
0.2758
0.7577
0.9814
0.2766

0.1721
0.7809
0.7654
0.8365
0.2829
0.7652
0.9819
0.2812



In [4]:
file = '/home/server/Projects/data/AKI/tabular_combined.csv'
df = pd.read_csv(file)

# -------------------- MLP CLASSIFICATION --------------------

from sklearn.neural_network import MLPClassifier

df_results = pd.DataFrame()
for test, train in tqdm(bootstrap_split(df)):
    X_test, y_test, y_binary_test = test
    X_train, y_train, y_binary_train = train


    arch = (8, 8, 4, 4)

    model = MLPClassifier(random_state=42, max_iter=1000, hidden_layer_sizes=arch)
    model.fit(X_train, y_binary_train)
    y_pred_binary = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    output_dict = performance_dict(y_binary_test, y_pred_binary, y_prob)



    if df_results.empty:
        df_results = pd.DataFrame(columns=output_dict.keys())
    df_results.loc[len(df_results)] = output_dict

print_confidence_intervals(df_results)
    

25it [1:09:08, 165.93s/it]

Precision
Sensitivity
Accuracy
rc_auc
pr_auc
Specificity
Negative Predictive Value
F1 Score

0.2018
0.5543
0.8407
0.7126
0.2054
0.8580
0.9684
0.2967

0.2064
0.5634
0.8446
0.7234
0.2139
0.8624
0.9689
0.3019

0.2111
0.5726
0.8484
0.7342
0.2224
0.8667
0.9695
0.3070



In [6]:
# -------------------- RANDOM FOREST CLASSIFICATION --------------------

from sklearn.ensemble import RandomForestClassifier

df_results = pd.DataFrame()
for test, train in tqdm(bootstrap_split(df)):
    X_test, y_test, y_binary_test = test
    X_train, y_train, y_binary_train = train



    model = RandomForestClassifier()
    model.fit(X_train, y_binary_train)
    y_pred_binary = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    output_dict = performance_dict(y_binary_test, y_pred_binary, y_prob)



    if df_results.empty:
        df_results = pd.DataFrame(columns=output_dict.keys())
    df_results.loc[len(df_results)] = output_dict

print_confidence_intervals(df_results)
    

0it [00:00, ?it/s]

25it [26:48, 64.35s/it]

Precision
Sensitivity
Accuracy
rc_auc
pr_auc
Specificity
Negative Predictive Value
F1 Score

0.5127
0.0852
0.9407
0.7784
0.2529
0.9947
0.9450
0.1463

0.5288
0.0883
0.9410
0.7818
0.2583
0.9950
0.9452
0.1512

0.5449
0.0913
0.9413
0.7853
0.2638
0.9953
0.9453
0.1561



In [7]:
# -------------------- LINEAR REGRESSION BASED CLASSIFICATION --------------------
from sklearn.linear_model import LinearRegression

df_results = pd.DataFrame()
for test, train in tqdm(bootstrap_split(df)):
    X_test, y_test, y_binary_test = test
    X_train, y_train, y_binary_train = train


    model = LinearRegression()
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_binary = y_pred > 0.3
    y_prob = sigmoid(y_pred - 0.3)
    output_dict = performance_dict(y_binary_test, y_pred_binary, y_prob)



    if df_results.empty:
        df_results = pd.DataFrame(columns=output_dict.keys())
    df_results.loc[len(df_results)] = output_dict

print_confidence_intervals(df_results)
    

2it [00:28, 14.31s/it]/home/server/Projects/VitalDB-Dimensionality-Reduction/preoperative_models/utils.py:12: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
7it [01:36, 13.49s/it]/home/server/Projects/VitalDB-Dimensionality-Reduction/preoperative_models/utils.py:12: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
16it [03:36, 13.73s/it]/home/server/Projects/VitalDB-Dimensionality-Reduction/preoperative_models/utils.py:12: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
25it [05:47, 13.88s/it]

Precision
Sensitivity
Accuracy
rc_auc
pr_auc
Specificity
Negative Predictive Value
F1 Score

0.0963
0.7381
0.5708
0.7187
0.1676
0.5597
0.9713
0.1705

0.0974
0.7461
0.5731
0.7240
0.1733
0.5621
0.9722
0.1723

0.0984
0.7540
0.5754
0.7293
0.1791
0.5645
0.9730
0.1741



In [8]:
# -------------------- K-NEAREST NEIGHBORS CLASSIFICATION --------------------
df_results = pd.DataFrame()
for test, train in tqdm(bootstrap_split(df)):
    X_test, y_test, y_binary_test = test
    X_train, y_train, y_binary_train = train


    model = KNeighborsClassifier(n_neighbors=200)
    model.fit(X_train, y_binary_train)
    y_pred_binary = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    output_dict = performance_dict(y_binary_test, y_pred_binary, y_prob)



    if df_results.empty:
        df_results = pd.DataFrame(columns=output_dict.keys())
    df_results.loc[len(df_results)] = output_dict

print_confidence_intervals(df_results)
    

25it [16:40, 40.01s/it]

Precision
Sensitivity
Accuracy
rc_auc
pr_auc
Specificity
Negative Predictive Value
F1 Score

0.1328
0.6339
0.7299
0.7556
0.2159
0.7355
0.9696
0.2198

0.1344
0.6407
0.7327
0.7586
0.2209
0.7386
0.9701
0.2222

0.1359
0.6475
0.7356
0.7615
0.2259
0.7417
0.9706
0.2245



In [9]:
# -------------------- LOGISTIC REGRESSION BASED CLASSIFICATION --------------------

df_results = pd.DataFrame()
for test, train in tqdm(bootstrap_split(df)):
    X_test, y_test, y_binary_test = test
    X_train, y_train, y_binary_train = train




    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_binary_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    output_dict = performance_dict(y_binary_test, y_pred, y_prob)





    if df_results.empty:
        df_results = pd.DataFrame(columns=output_dict.keys())
    df_results.loc[len(df_results)] = output_dict

print_confidence_intervals(df_results)
    

25it [07:02, 16.91s/it]

Precision
Sensitivity
Accuracy
rc_auc
pr_auc
Specificity
Negative Predictive Value
F1 Score

0.1361
0.6581
0.7298
0.7712
0.2303
0.7339
0.9714
0.2256

0.1375
0.6649
0.7316
0.7748
0.2367
0.7358
0.9720
0.2279

0.1389
0.6717
0.7334
0.7783
0.2431
0.7378
0.9725
0.2301

